In [1]:
import pandas as pd
import openai
from dotenv import load_dotenv
from openai import OpenAI
import os
import glob
import json
import sys
sys.path.append(r'c:\Users\ricca\Documents\Polimi\tesi\code')
import utils
import warnings
warnings.filterwarnings('ignore')

import importlib
importlib.reload(utils)

<module 'utils' from 'c:\\Users\\ricca\\Documents\\Polimi\\tesi\\code\\utils.py'>

In [2]:
load_dotenv()


True

qui devo caricare le seguenti query:
DS1-E-0005
DS1-E-0016
DS1-E-0045
DS1-E-0049
DS1-E-0065

In [3]:
# Definisci la lista degli ID da filtrare
codici = [
	"DS1-E-0005",
	"DS1-E-0016",
	"DS1-E-0045",
	"DS1-E-0049",
	"DS1-E-0065"
]

# Filtra le query per gli id specificati in 'codici'
# Carica il file TSV e filtra le query per gli id specificati in 'codici'
df_queries = pd.read_csv('benchmark/data_search_2_e_train_topics.tsv', sep='\t', header=None)
filtered_queries = df_queries[df_queries[df_queries.columns[0]].isin(codici)]
filtered_queries = dict(zip(filtered_queries[filtered_queries.columns[0]], filtered_queries[filtered_queries.columns[1]]))
# Visualizza le query filtrate
print(filtered_queries)

{'DS1-E-0005': 'births by month', 'DS1-E-0016': 'trend number police officer 1945', 'DS1-E-0045': 'recent gender birth rate in us', 'DS1-E-0049': 'united states births and deaths per second', 'DS1-E-0065': 'percentage of americans with drivers license'}


In [ ]:
results = {}

for query_id, query_text in filtered_queries.items():
    prompt = f"""You are provided with the following keywords query: {query_text}
    Use the keywords to generate a search query for a search engine.
    The search query should be in natural language and should be of high complexity and length. 
    This is an example of a search query: "Find me a dataset about the impact of climate change on agriculture in Europe"

    The answer should be a single string, without any additional text or explanation.
"""
    response = utils.call_openai_api(prompt, "o1-mini").choices[0].message.content
    results[query_id] = response

print(results)

{'DS1-E-0005': '"Find comprehensive statistical data and detailed analyses on the number of births categorized by each calendar month, including historical trends, regional variations, seasonal patterns, and potential correlations with socio-economic, environmental, or healthcare factors over the past several decades."', 'DS1-E-0016': '"Provide an in-depth analysis of the statistical trends and numerical data concerning the number of police officers employed across different regions in the year 1945, including the socio-economic factors influencing these figures and the impact of global events on law enforcement personnel during that period."', 'DS1-E-0045': '"Conduct an in-depth search for the most recent and comprehensive statistical reports, datasets, and scholarly analyses pertaining to gender-specific birth rates in the United States, encompassing regional variations, temporal trends over the past several years, potential socio-economic and cultural factors influencing these rates

In [8]:
def get_summary(query):

    prompt_to_summary = f"""
    Generate a background document from web to answer the given question. {query}

    No additional text is needed, just the background document.
    """

    return utils.call_openai_api(prompt_to_summary, "o1-mini").choices[0].message.content

In [9]:
summaries = {}

for query_id, query_text in results.items():
    summary = get_summary(query_text)
    summaries[query_id] = summary

In [10]:
def get_instructions(query, summary):

    prompt_to_instructions = f"""
    Address the following query based on the contexts provided. Identify any missing information or areas
    requiring validation, especially if time-sensitive data is involved. Then, formulate several specific search engine
    queries to acquire or validate the necessary knowledge

    Query: {query}
    Context: {summary}

    Output the queries you generate in a numbered list, no other text is required.
    """

    return utils.call_openai_api(prompt_to_instructions, "o1-mini").choices[0].message.content

In [11]:
instructions = {}

# Initialize instructions_dict as a list before the loop
instructions_dict = []

for query_id, query_text in results.items():
    summary = summaries[query_id]
    instructions = get_instructions(query_text, summary)

    print(instructions)

    if isinstance(instructions, str):
        instructions_list = [instr.strip().strip('"').strip("'") for instr in instructions.split('\n') if instr.strip()]
    else:
        instructions_list = instructions

    for idx, instr in enumerate(instructions_list, 1):
        unique_code = f"{query_id}_Q{idx:02d}"
        instructions_dict.append({
            'unique_code': unique_code,
            'query_id': query_id,
            'instruction': instr
        })
print(instructions_dict)


1. "Comprehensive monthly birth rates statistics by country from 1950 to 2023"
2. "Historical trends in monthly birth rates and significant socio-economic events impact"
3. "Regional variations in birth rates by month across different continents"
4. "Seasonal patterns in birth rates and correlation with climate data"
5. "Impact of economic recessions on monthly birth rate distributions"
6. "Correlation between healthcare access and monthly birth rates over decades"
7. "Urban vs rural monthly birth rate differences and influencing factors"
8. "Effects of natural disasters on monthly birth rates in affected regions"
9. "Influence of public health policies on the timing of childbirth"
10. "Statistical analyses of monthly birth rates and environmental pollution levels"
11. "Monthly birth rates and their association with educational and employment opportunities"
12. "Cultural norms and their impact on monthly distribution of births"
13. "Data sources for longitudinal studies on monthly birt

In [12]:
import pickle

with open('instructions_dict_nl.pkl', 'wb') as f:
    pickle.dump(instructions_dict, f)